# Persistent Memory and Derived Context: A Two-Layer Pattern for Agents

Companion notebook for the Oracle AI Database article *Persistent Memory and Derived Context: A Two-Layer Pattern for Agents*. It runs on the schema from the previous articles, From Prompt to Persistence [part 1](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-1-designing-multi-tenant-agent-memory-schemas-for-saas) and [part 2](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-2-putting-the-multi-tenant-agent-memory-schema-to-work), unchanged. Their companion notebook is [multitenant_schema_walkthrough.ipynb](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/8c9ed028b1bea545f2cbefe057470a0294bf29b7/notebooks/multitenant_schema_walkthrough.ipynb).

The boundary in this article is logical. Persistent memory is what's true; derived context is that same knowledge shaped for fast retrieval. In a converged store the two share a row: `content` is canonical, the `embedding` column is derived from it. Provenance runs one way, from the embedding back to the content, and because they live in the same row, one ACID write keeps them together. There's no separate derived-layer table. The derived layer is a column.

The notebook runs the whole pattern against a live database. It builds the schema, writes a fact and its embedding in one statement, then corrects them in one statement and measures that the vector still tracks the fact. It chunks a document into the knowledge-base tables and rebuilds those chunks from the source. And it walks the sync strategies that keep derived context in step with canonical.

## Prerequisites

- Oracle AI Database at `localhost:1521/FREEPDB1` (or set `DB_DSN`), with a non-zero `vector_memory_size`.
- A user with `DB_DEVELOPER_ROLE`, `CREATE SESSION`, `CREATE TABLE`, `CREATE MATERIALIZED VIEW`, `EXECUTE ON DBMS_RLS`.
- The ONNX model `ALL_MINILM_L12_V2` (384-dim) loaded.
- A `.env` with `DB_USER`, `DB_PASSWORD`, `DB_DSN`.

Vectors are `VECTOR(384, FLOAT32)` to match `ALL_MINILM_L12_V2`.

## 1. Environment & connection

In [1]:
import os, hashlib, uuid

import oracledb
from dotenv import load_dotenv

load_dotenv()

# Return CLOB columns (content, summary, chunk text) as str, not LOB locators, so the
# SUBSTR previews below print as text.
oracledb.defaults.fetch_lobs = False

DB_USER     = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_DSN      = os.getenv("DB_DSN", "localhost:1521/FREEPDB1")

conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cur = conn.cursor()
# Pin the session to UTC — valid_from/valid_until/created_at are plain TIMESTAMP written
# from SYSTIMESTAMP (DB tz = UTC); a different session tz would misjudge superseded rows.
cur.execute("ALTER SESSION SET TIME_ZONE = 'UTC'")

def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("Oracle version:", conn.version)

Oracle version: 23.26.1.0.0


## 2. The schema: canonical content and derived embedding, same row

We rebuild the part of the previous articles' schema this walkthrough uses: `entity_memory` and `summarization_memory`, the append-only `conversation_memory`, and the knowledge-base pair. Each semantically-retrieved type carries an inline `embedding VECTOR(384, FLOAT32)` next to its canonical text. `conversation_memory` is the exception on purpose. It's the high-volume event stream, so it carries no embedding at all. Its derived context gets built separately, in section 10.

In [2]:
try:
    cur.execute("DROP MATERIALIZED VIEW active_fact_retrieval")
except oracledb.DatabaseError:
    pass

TABLES = ["knowledge_base_chunk", "knowledge_base_document",
          "summarization_memory", "conversation_memory", "entity_memory"]
for t in TABLES:
    try:
        cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")   # drops RLS policies with it
    except oracledb.DatabaseError:
        pass
print("Dropped MV + tables (if they existed).")

Dropped MV + tables (if they existed).


In [3]:
# entity_memory — canonical fact + inline (derived) embedding on the same row.
cur.execute("""
CREATE TABLE entity_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),
  subject         VARCHAR2(256) NOT NULL,
  predicate       VARCHAR2(64)  NOT NULL,
  content         CLOB,                       -- canonical fact text
  content_hash    VARCHAR2(64)  NOT NULL,
  content_json    JSON,
  embedding       VECTOR(384, FLOAT32),       -- derived from content, same row
  confidence      NUMBER(3,2)   NOT NULL,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")
cur.execute("""
CREATE INDEX idx_entity_scope_active
  ON entity_memory (tenant_id, user_id, agent_id, thread_id, deleted_at, valid_until)
""")
cur.execute("CREATE INDEX idx_entity_subject ON entity_memory (tenant_id, subject)")
cur.execute("""
CREATE VECTOR INDEX idx_entity_embedding ON entity_memory (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE WITH TARGET ACCURACY 95
""")
print("entity_memory created (inline embedding + vector index).")

entity_memory created (inline embedding + vector index).


In [4]:
# summarization_memory — distilled episode + inline embedding.
cur.execute("""
CREATE TABLE summarization_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),
  task_type       VARCHAR2(64)  NOT NULL,
  title           VARCHAR2(256) NOT NULL,
  summary         CLOB,
  outcome         VARCHAR2(64),
  embedding       VECTOR(384, FLOAT32),       -- derived from summary, same row
  confidence      NUMBER(3,2)   NOT NULL,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  completed_at    TIMESTAMP     NOT NULL,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")
cur.execute("""
CREATE VECTOR INDEX idx_summ_embedding ON summarization_memory (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE WITH TARGET ACCURACY 95
""")
print("summarization_memory created (inline embedding + vector index).")

summarization_memory created (inline embedding + vector index).


In [5]:
# conversation_memory — high-volume raw event stream, NO inline embedding by design.
cur.execute("""
CREATE TABLE conversation_memory (
  event_id    VARCHAR2(64)  PRIMARY KEY,
  tenant_id   VARCHAR2(64)  NOT NULL,
  user_id     VARCHAR2(64),
  thread_id   VARCHAR2(64),
  run_id      VARCHAR2(64),
  turn_index  NUMBER,
  event_type  VARCHAR2(32),
  role        VARCHAR2(32),
  payload     JSON,
  created_at  TIMESTAMP     NOT NULL
)
""")

# knowledge_base_document — a POINTER (source_uri) + metadata; the bytes stay in object storage.
cur.execute("""
CREATE TABLE knowledge_base_document (
  id          VARCHAR2(64)   PRIMARY KEY,
  tenant_id   VARCHAR2(64)   NOT NULL,
  collection  VARCHAR2(128)  NOT NULL,
  title       VARCHAR2(512)  NOT NULL,
  source_uri  VARCHAR2(1024),
  source_type VARCHAR2(64)   NOT NULL,
  ingested_at TIMESTAMP      NOT NULL,
  ingested_by VARCHAR2(64)   NOT NULL,
  version     NUMBER(10)     NOT NULL,
  valid_from  TIMESTAMP      NOT NULL,
  valid_until TIMESTAMP,
  created_at  TIMESTAMP      NOT NULL,
  deleted_at  TIMESTAMP
)
""")

# knowledge_base_chunk — derived chunks of a document, each with an inline embedding.
cur.execute("""
CREATE TABLE knowledge_base_chunk (
  id          VARCHAR2(64)  PRIMARY KEY,
  tenant_id   VARCHAR2(64)  NOT NULL,
  document_id VARCHAR2(64)  NOT NULL,
  chunk_index NUMBER(10)    NOT NULL,
  content     CLOB          NOT NULL,
  embedding   VECTOR(384, FLOAT32),
  created_at  TIMESTAMP     NOT NULL,
  deleted_at  TIMESTAMP,
  CONSTRAINT fk_kb_chunk_doc FOREIGN KEY (document_id) REFERENCES knowledge_base_document (id)
)
""")
cur.execute("""
CREATE UNIQUE INDEX idx_kb_chunk_doc_idx
  ON knowledge_base_chunk (tenant_id, document_id, chunk_index)
""")
cur.execute("""
CREATE VECTOR INDEX idx_kb_chunk_embedding ON knowledge_base_chunk (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE WITH TARGET ACCURACY 95
""")
print("conversation_memory + knowledge_base_document + knowledge_base_chunk created.")

conversation_memory + knowledge_base_document + knowledge_base_chunk created.


## 3. Tenant isolation and session context

Row-level security carries over from the previous articles, unchanged. One policy function returns `tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')`, attached to every tenant-scoped table. `update_check => TRUE` stops a session from writing rows for another tenant.

In [6]:
cur.execute("""
CREATE OR REPLACE FUNCTION memory_tenant_policy(
  schema_in IN VARCHAR2, table_in IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
  RETURN q'[
    tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')
  ]';
END;
""")

cur.execute("""
BEGIN
  FOR rec IN (SELECT table_name FROM user_tables
               WHERE table_name LIKE '%_MEMORY'
                  OR table_name LIKE 'KNOWLEDGE_BASE%')
  LOOP
    BEGIN
      DBMS_RLS.ADD_POLICY(
        object_schema   => USER,
        object_name     => rec.table_name,
        policy_name     => rec.table_name || '_tenant_pol',
        policy_function => 'memory_tenant_policy',
        statement_types => 'SELECT,INSERT,UPDATE,DELETE',
        update_check    => TRUE);
    EXCEPTION WHEN OTHERS THEN
      IF SQLCODE = -28101 THEN NULL;   -- already exists; idempotent
      ELSE RAISE;
      END IF;
    END;
  END LOOP;
END;
""")

cur.execute("""
SELECT object_name, policy_name FROM user_policies
 WHERE object_name LIKE '%_MEMORY' OR object_name LIKE 'KNOWLEDGE_BASE%'
 ORDER BY object_name
""")
for row in cur:
    print(f"  {row[0]:<24} <- {row[1]}")
conn.commit()

  CONVERSATION_MEMORY      <- CONVERSATION_MEMORY_TENANT_POL
  ENTITY_MEMORY            <- ENTITY_MEMORY_TENANT_POL
  KNOWLEDGE_BASE_CHUNK     <- KNOWLEDGE_BASE_CHUNK_TENANT_POL
  KNOWLEDGE_BASE_DOCUMENT  <- KNOWLEDGE_BASE_DOCUMENT_TENANT_POL
  SUMMARIZATION_MEMORY     <- SUMMARIZATION_MEMORY_TENANT_POL


In [7]:
cur.execute("""
CREATE OR REPLACE PACKAGE set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant_id IN VARCHAR2);
END;
""")
cur.execute("""
CREATE OR REPLACE PACKAGE BODY set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant_id IN VARCHAR2) IS
  BEGIN
    DBMS_SESSION.SET_CONTEXT('memory_ctx', 'tenant_id', p_tenant_id);
  END;
END;
""")
cur.execute("CREATE OR REPLACE CONTEXT memory_ctx USING set_memory_ctx")

TENANT_ACME = "acme"
USER_ID     = "user:jane@acme.com"

cur.callproc("set_memory_ctx.set_tenant", [TENANT_ACME])
cur.execute("SELECT SYS_CONTEXT('memory_ctx','tenant_id') FROM dual")
print("Active tenant:", cur.fetchone()[0])

Active tenant: acme


## 4. Content is canonical, embedding is derived, one row and one write

The article opens with a benchmark fact: Helios-RAG scores 0.92 nDCG@10 on BEIR. We seed it as an `entity_memory` row at version 1. The `content` is the canonical fact; the `embedding` is computed from that content by the database in the same `INSERT`. They enter together. We also seed the source conversation event and the paper's `knowledge_base_document` row, which the fact cites through `content_json`.

In [8]:
# The paper the fact came from (a pointer + metadata; the PDF stays in object storage).
kb_doc_id = new_id("kbd")
cur.execute("""
INSERT INTO knowledge_base_document (id, tenant_id, collection, title, source_uri, source_type,
                                     ingested_at, ingested_by, version, valid_from, created_at)
VALUES (:id, :t, 'benchmarks',
        'Helios-RAG: A Retrieval-Augmented Model for Open-Domain QA (with erratum)',
        's3://acme-kb/papers/helios-rag.pdf', 'pdf', SYSTIMESTAMP, 'admin:acme', 1,
        SYSTIMESTAMP, SYSTIMESTAMP)
""", id=kb_doc_id, t=TENANT_ACME)

# The source event that asserted the fact.
ev1 = new_id("evt")
cur.execute("""
INSERT INTO conversation_memory (event_id, tenant_id, user_id, thread_id, run_id,
                                 turn_index, event_type, role, payload, created_at)
VALUES (:e, :t, :u, 'thread_demo', 'run_demo', 1, 'model_msg', 'assistant',
        JSON('{ "text": "Helios-RAG scores 0.92 nDCG@10 on BEIR." }'), SYSTIMESTAMP)
""", e=ev1, t=TENANT_ACME, u=USER_ID)

# The canonical fact + its inline embedding, in one statement.
ent_v1 = new_id("ent")
content_v1 = "Helios-RAG achieves 0.92 nDCG@10 on the BEIR benchmark."
cur.execute("""
INSERT INTO entity_memory (
  id, tenant_id, user_id, subject, predicate, content, content_hash, content_json,
  embedding, confidence, written_by, source_event_id, version, valid_from, created_at
) VALUES (
  :id, :t, :u, 'model:helios-rag', 'beir_ndcg10', :c, :h,
  JSON('{ "entity_type": "benchmark_result", "source_document_id": "' || :doc || '" }'),
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :c2 AS DATA),
  0.95, 'agent:research_v1', :ev, 1, SYSTIMESTAMP, SYSTIMESTAMP
)
""", id=ent_v1, t=TENANT_ACME, u=USER_ID, c=content_v1, h=sha256(content_v1),
     doc=kb_doc_id, c2=content_v1, ev=ev1)
conn.commit()

cur.execute("""
SELECT version, SUBSTR(content,1,52) AS content,
       CASE WHEN embedding IS NULL THEN 'no' ELSE 'yes' END AS has_embedding
  FROM entity_memory WHERE id = :id
""", id=ent_v1)
r = cur.fetchone()
print(f"Seeded fact v{r[0]}: {r[1]!r}  | inline embedding present: {r[2]}")

Seeded fact v1: 'Helios-RAG achieves 0.92 nDCG@10 on the BEIR benchma'  | inline embedding present: yes


## 5. The transactional supersede

Six weeks on, the score is corrected to 0.78. In a polyglot stack the fact lives in one store and its vector in another, so a correction is two writes that can diverge, and the gap between them is where the opening scenario's drift comes from. In a converged store it's one statement. The `INSERT` carries the unchanged columns forward from the old row and recomputes the embedding from the corrected text. When the commit returns, no version of reality has the fact at 0.78 while its embedding still says 0.92. They're the same row, and ACID moved them together.

In [9]:
corr_ev = new_id("evt")
cur.execute("""
INSERT INTO conversation_memory (event_id, tenant_id, user_id, thread_id, run_id,
                                 turn_index, event_type, role, payload, created_at)
VALUES (:e, :t, :u, 'thread_demo', 'run_demo', 2, 'model_msg', 'assistant',
        JSON('{ "text": "Correction: Helios-RAG BEIR score revised to 0.78." }'), SYSTIMESTAMP)
""", e=corr_ev, t=TENANT_ACME, u=USER_ID)

ent_v2 = new_id("ent")
content_v2 = "Helios-RAG achieves 0.78 nDCG@10 on the BEIR benchmark (corrected)."
cur.execute("""
INSERT INTO entity_memory (
  id, tenant_id, user_id, agent_id, thread_id, subject, predicate, content, content_hash,
  content_json, embedding, confidence, written_by, source_event_id, version, valid_from, created_at
)
SELECT :new_id, tenant_id, user_id, agent_id, thread_id, subject, predicate,
       :c, :h, content_json,
       VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :c2 AS DATA),
       confidence, written_by, :ev, version + 1, SYSTIMESTAMP, SYSTIMESTAMP
FROM   entity_memory WHERE id = :old_id
""", new_id=ent_v2, c=content_v2, h=sha256(content_v2), c2=content_v2, ev=corr_ev, old_id=ent_v1)

cur.execute("""
UPDATE entity_memory SET valid_until = SYSTIMESTAMP, superseded_by = :new_id
 WHERE id = :old_id AND valid_until IS NULL
""", new_id=ent_v2, old_id=ent_v1)
conn.commit()

cur.execute("""
SELECT version, SUBSTR(content,1,54) AS content, valid_until
  FROM entity_memory
 WHERE subject = 'model:helios-rag' AND predicate = 'beir_ndcg10'
 ORDER BY version
""")
print("Supersession chain (content + inline embedding moved together):")
for r in cur:
    print(f"  v{r[0]}  {r[1]!r}  {'<active>' if r[2] is None else 'retired'}")

Supersession chain (content + inline embedding moved together):
  v1  'Helios-RAG achieves 0.92 nDCG@10 on the BEIR benchmark'  retired
  v2  'Helios-RAG achieves 0.78 nDCG@10 on the BEIR benchmark'  <active>


## 6. No drift, by construction

The embedding was recomputed from the corrected content in the same write, so the in-force row's embedding reflects the corrected 0.78. We can measure it. The cosine distance between the stored embedding and a fresh embedding of the current content is about zero, while the distance to a fresh embedding of the old content is larger. The vector tracks the fact. No stale-embedding path is left to retrieve.

In [10]:
cur.execute("""
SELECT VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :cur AS DATA), COSINE) AS d_current,
       VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :old AS DATA), COSINE) AS d_old
  FROM entity_memory
 WHERE subject = 'model:helios-rag' AND predicate = 'beir_ndcg10'
   AND valid_until IS NULL AND deleted_at IS NULL
""", cur=content_v2, old=content_v1)
d_current, d_old = cur.fetchone()
print(f"stored embedding vs CURRENT content (0.78): distance = {d_current:.6f}  (~0 -> tracks the fact)")
print(f"stored embedding vs OLD content     (0.92): distance = {d_old:.6f}  (larger -> not the stale vector)")
print("no drift:", d_current < d_old)

stored embedding vs CURRENT content (0.78): distance = 0.000000  (~0 -> tracks the fact)
stored embedding vs OLD content     (0.92): distance = 0.016073  (larger -> not the stale vector)
no drift: True


## 7. Summaries carry an inline embedding

`summarization_memory` carries an inline embedding like the other semantic types, so a finished summary is written with its vector in one statement. What sets summaries apart is the sync strategy after that. A summary is an episodic snapshot of completed work, so when the source it was distilled from moves on, you rebuild the summary on a schedule, and sometimes you leave it pinned on purpose. Here we write one finished summary with its embedding.

In [11]:
sum_id = new_id("sum")
summary = ("Evaluated Helios-RAG for the customer's retrieval stack. On BEIR it lands at "
           "0.78 nDCG@10 after the contamination correction, below the 0.85 bar the customer "
           "set, so we recommended against adopting it for the compliance corpus.")
cur.execute("""
INSERT INTO summarization_memory (
  id, tenant_id, user_id, task_type, title, summary, outcome,
  embedding, confidence, written_by, source_event_id, version, valid_from, completed_at, created_at
) VALUES (
  :id, :t, :u, 'model_eval', 'Helios-RAG evaluation for compliance corpus', :s, 'recommend_against',
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :s2 AS DATA),
  0.9, 'agent:research_v1', :ev, 1, SYSTIMESTAMP, SYSTIMESTAMP, SYSTIMESTAMP
)
""", id=sum_id, t=TENANT_ACME, u=USER_ID, s=summary, s2=summary, ev=corr_ev)
conn.commit()

cur.execute("""
SELECT title, outcome, CASE WHEN embedding IS NULL THEN 'no' ELSE 'yes' END
  FROM summarization_memory WHERE id = :id
""", id=sum_id)
r = cur.fetchone()
print(f"Summary inserted inline: {r[0]!r}  outcome={r[1]!r}  embedding present: {r[2]}")

Summary inserted inline: 'Helios-RAG evaluation for compliance corpus'  outcome='recommend_against'  embedding present: yes


## 8. Chunking is a knowledge-base operation

Chunking splits a document into passages and embeds each one, and the schema has two tables for it: `knowledge_base_document`, the pointer, and `knowledge_base_chunk`, the chunked text plus an inline embedding per chunk. That's where `DBMS_VECTOR_CHAIN` earns its place, and it's the table the article's rebuild query writes to. Two details make that query run. `UTL_TO_CHUNKS` returns a collection of JSON documents, so the chunk text is `JSON_VALUE(ct.column_value, '$.chunk_data')`. And each chunk needs its own key, so we generate a per-row `SYS_GUID()` and number the chunks with `ROW_NUMBER()`. The canonical source is the document in object storage, referenced by `source_uri`; here `doc_text` stands in for its fetched text.

In [12]:
# Stand-in for the document's fetched text (in production, read from source_uri).
doc_text = (
  "Helios-RAG is a retrieval-augmented model for open-domain question answering. "
  "The original paper reported 0.92 nDCG@10 on the BEIR benchmark, averaged across its 14 "
  "retrieval tasks. A later erratum traced that figure to a data-contamination bug: several "
  "BEIR test passages had leaked into the training corpus, inflating retrieval scores. After "
  "removing the leaked passages and re-running the evaluation, the corrected score is 0.78 "
  "nDCG@10. The authors attribute the gap primarily to the MS MARCO and HotpotQA splits, "
  "where contamination was heaviest. The architecture and training recipe are unchanged; "
  "only the benchmark number was revised."
)

# Chunk-and-embed pipeline: chunk the source, embed each chunk, land in knowledge_base_chunk.
INGEST_CHUNKS = """
INSERT INTO knowledge_base_chunk (id, tenant_id, document_id, chunk_index, content, embedding, created_at)
WITH chunks AS (
  SELECT JSON_VALUE(ct.column_value, '$.chunk_data')                AS chunk_data,
         JSON_VALUE(ct.column_value, '$.chunk_id' RETURNING NUMBER) AS chunk_id
  FROM   TABLE(DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS(:doc_text, JSON(:params))) ct
)
SELECT 'kbc_' || LOWER(RAWTOHEX(SYS_GUID())),        -- unique id per chunk
       :tenant_id, :doc_id,
       ROW_NUMBER() OVER (ORDER BY chunk_id),        -- chunk_index 1..N
       chunk_data,
       VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING chunk_data AS DATA),
       SYS_EXTRACT_UTC(SYSTIMESTAMP)
FROM chunks
"""
cur.execute(INGEST_CHUNKS, doc_text=doc_text,
            params='{"by":"words","max":80,"overlap":10}', tenant_id=TENANT_ACME, doc_id=kb_doc_id)
conn.commit()

cur.execute("""
SELECT chunk_index, SUBSTR(content,1,58) FROM knowledge_base_chunk
 WHERE document_id = :d ORDER BY chunk_index
""", d=kb_doc_id)
rows = cur.fetchall()
print(f"Ingested {len(rows)} chunks (each with an inline embedding):")
for r in rows:
    print(f"  [{r[0]}] {r[1]!r}")

Ingested 2 chunks (each with an inline embedding):
  [1] 'Helios-RAG is a retrieval-augmented model for open-domain '
  [2] 'passages and re-running the evaluation, the corrected scor'


## 9. The rebuild path is the test

The question that separates two layers from two tables: can you delete the whole derived layer and rebuild it from canonical alone? For the knowledge base, the derived layer is the chunks and their embeddings, and the canonical source is the document. Delete the chunks and re-ingest from the source, changing the chunk strategy while you're at it, because a new chunk size or a new embedding model is the same operation. The `knowledge_base_document` row never moves.

In [13]:
cur.execute("SELECT COUNT(*) FROM knowledge_base_chunk WHERE document_id = :d", d=kb_doc_id)
before = cur.fetchone()[0]

cur.execute("DELETE FROM knowledge_base_chunk WHERE document_id = :d", d=kb_doc_id)
print(f"Deleted the derived chunks ({before} rows).")

# Rebuild from canonical, with a finer chunk strategy.
cur.execute(INGEST_CHUNKS, doc_text=doc_text,
            params='{"by":"words","max":40,"overlap":8}', tenant_id=TENANT_ACME, doc_id=kb_doc_id)
conn.commit()

cur.execute("SELECT COUNT(*) FROM knowledge_base_chunk WHERE document_id = :d", d=kb_doc_id)
after = cur.fetchone()[0]
cur.execute("SELECT title, version FROM knowledge_base_document WHERE id = :d", d=kb_doc_id)
doc = cur.fetchone()
print(f"Rebuilt with finer chunks (max=80 -> {before} chunks; max=40 -> {after} chunks).")
print(f"Canonical document unchanged: {doc[0]!r} (v{doc[1]}).")

Deleted the derived chunks (2 rows).
Rebuilt with finer chunks (max=80 -> 2 chunks; max=40 -> 4 chunks).
Canonical document unchanged: 'Helios-RAG: A Retrieval-Augmented Model for Open-Domain QA (with erratum)' (v1).


## 10. Three ways to keep the two layers in sync

Once derived context can drift from canonical, the question is how fast you close the gap. You pick a strategy per memory type, weighing how costly a stale read is against how often the canonical data changes. A transactional rebuild happens in the same write, which is what an entity fact needs when a corrected number has to be right immediately. A scheduled refresh runs on a cadence, and it suits pre-joined materialized views and episodic summaries that tolerate an hour of lag. Lazy leaves the work until a read misses, which is all a high-volume conversation rollup requires. A converged store makes the transactional write cheap, since the embedding is computed in the same database in the same transaction, so you reserve the other two for the cases where cost is the constraint.

### 10a. Scheduled: a materialized view

`active_fact_retrieval` pre-joins in-force facts to their source documents and refreshes hourly. The `REFRESH ... NEXT ... INTERVAL '1' HOUR` clause is the scheduled strategy, written in DDL.

> A materialized view can't be built over an RLS-protected table: its background refresh has no tenant context, so Oracle raises **ORA-30372**. An MV is a deployment-level projection, so we build it outside the tenant predicate. Detach the base-table policies, create the view, reattach them. In production you'd scope the projection per tenant.

In [14]:
MV_DDL = """
CREATE MATERIALIZED VIEW active_fact_retrieval
  REFRESH COMPLETE
  START WITH SYS_EXTRACT_UTC(SYSTIMESTAMP)
  NEXT SYS_EXTRACT_UTC(SYSTIMESTAMP) + INTERVAL '1' HOUR
AS
SELECT e.id, e.tenant_id, e.subject, e.predicate, e.content, e.embedding,
       d.title, d.source_uri
FROM   entity_memory e
LEFT   JOIN knowledge_base_document d
  ON   d.id = JSON_VALUE(e.content_json, '$.source_document_id')
WHERE  e.valid_until IS NULL
  AND  e.deleted_at  IS NULL
"""

# The article's DDL, as written, over the RLS-protected base tables.
try:
    cur.execute(MV_DDL)
    print("Created (no policy on the base tables).")
except oracledb.DatabaseError as e:
    print("Article DDL under RLS ->", str(e).splitlines()[0])

# Build outside the tenant predicate: detach base policies, create, reattach.
def _drop_pol(tbl):
    cur.execute(f"""BEGIN DBMS_RLS.DROP_POLICY(USER, '{tbl}', '{tbl}_tenant_pol');
                    EXCEPTION WHEN OTHERS THEN NULL; END;""")
def _add_pol(tbl):
    cur.execute(f"""BEGIN DBMS_RLS.ADD_POLICY(object_schema=>USER, object_name=>'{tbl}',
                    policy_name=>'{tbl}_tenant_pol', policy_function=>'memory_tenant_policy',
                    statement_types=>'SELECT,INSERT,UPDATE,DELETE', update_check=>TRUE);
                    EXCEPTION WHEN OTHERS THEN IF SQLCODE=-28101 THEN NULL; ELSE RAISE; END IF; END;""")

for tbl in ("ENTITY_MEMORY", "KNOWLEDGE_BASE_DOCUMENT"):
    _drop_pol(tbl)
cur.execute(MV_DDL)
for tbl in ("ENTITY_MEMORY", "KNOWLEDGE_BASE_DOCUMENT"):
    _add_pol(tbl)
conn.commit()

cur.execute("SELECT subject, predicate, SUBSTR(content,1,40), title FROM active_fact_retrieval")
print("\nactive_fact_retrieval (scheduled derived projection):")
for r in cur:
    print(f"  {r[0]}/{r[1]}  {r[2]!r}  <- {r[3]!r}")

Article DDL under RLS -> ORA-30372: fine grain access policy conflicts with materialized view

active_fact_retrieval (scheduled derived projection):
  model:helios-rag/beir_ndcg10  'Helios-RAG achieves 0.78 nDCG@10 on the '  <- 'Helios-RAG: A Retrieval-Augmented Model for Open-Domain QA (with erratum)'


### 10b. Lazy: conversation rollups

`conversation_memory` carries no inline embedding on purpose. It's the high-volume event stream, and embedding every logged turn would be wasteful. Its derived context, a semantic rollup over recent turns, is computed asynchronously and rebuilt on a read miss. The staleness that creates is bounded and intentional, a spend decision.

In [15]:
# The raw stream exists and is queryable; it just has no derived vector on the row.
cur.execute("SELECT COUNT(*) FROM conversation_memory")
n = cur.fetchone()[0]
cur.execute("""
SELECT COUNT(*) FROM user_tab_columns
 WHERE table_name = 'CONVERSATION_MEMORY' AND column_name = 'EMBEDDING'
""")
has_vec = cur.fetchone()[0]
print(f"conversation_memory: {n} event(s), inline embedding column present: {'yes' if has_vec else 'no'}")
print("Its derived rollup is built lazily/asynchronously into a separate structure (illustrative):")
print("  CREATE TABLE conversation_rollup (... embedding VECTOR(384, FLOAT32) ...);")
print("  -- populated on read miss over recent turns, not on every event insert")

conversation_memory: 2 event(s), inline embedding column present: no
Its derived rollup is built lazily/asynchronously into a separate structure (illustrative):
  CREATE TABLE conversation_rollup (... embedding VECTOR(384, FLOAT32) ...);
  -- populated on read miss over recent turns, not on every event insert


## 11. Cleanup

Drop everything this notebook created: the materialized view, the policies, the tables, and the helper objects. Skip this cell to keep the seeded data around.

In [16]:
try:
    cur.execute("DROP MATERIALIZED VIEW active_fact_retrieval")
except oracledb.DatabaseError:
    pass

cur.execute("""
BEGIN
  FOR rec IN (SELECT object_name, policy_name FROM user_policies
               WHERE object_name LIKE '%_MEMORY' OR object_name LIKE 'KNOWLEDGE_BASE%')
  LOOP
    BEGIN
      DBMS_RLS.DROP_POLICY(USER, rec.object_name, rec.policy_name);
    EXCEPTION WHEN OTHERS THEN NULL;
    END;
  END LOOP;
END;
""")

for t in ["knowledge_base_chunk", "knowledge_base_document",
          "summarization_memory", "conversation_memory", "entity_memory"]:
    try:
        cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")
    except oracledb.DatabaseError:
        pass

for stmt in ["DROP PACKAGE set_memory_ctx", "DROP CONTEXT memory_ctx",
             "DROP FUNCTION memory_tenant_policy"]:
    try:
        cur.execute(stmt)
    except oracledb.DatabaseError:
        pass

conn.commit()
conn.close()
print("Cleanup complete.")

Cleanup complete.
